In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

In [ ]:
from abstractgraph.graphs import AbstractGraph
from abstractgraph.display import display, display_graph, display_mappings, display_decomposition_graph, decomposition_to_graph
from abstractgraph.operators import *

def draw(graph, df, nbits=10):
    display_decomposition_graph(decomposition_to_graph(df))
    ag = AbstractGraph(graph=graph).create_default_image_node().update()
    ag = df(ag)
    ag.update()
    #print(ag)
    display(ag, size=(12,6))
    display_mappings(ag)

In [ ]:
USE_EXAMPLE = 'chem'

if USE_EXAMPLE == 'random':
    from abstractgraph import RandomGraphConstructor
    graph = RandomGraphConstructor(integers_range=12, instance_size=40, alphabet_size=4).sample(1)
else:
    smi = 'CC1NC(OC2CC(O)C3(CO)C4C(O)CC5(C)C(C6CNC(=O)C6)CCC5(O)C4CCC3(O)C2)C(O)C(O)C1O'
    from abstractgraph_graphicalizer.chem import MoleculeGraphicalizer, draw_molecule
    graph = MoleculeGraphicalizer().transform([smi])[0]
    draw_molecule(graph)

print(graph)

display_graph(graph)

In [ ]:
cycle_df = compose(filter_by_number_of_nodes(number_of_nodes=(5,6)), cycle())
cycle_with_N_df = compose(filter_by_node_label(must_have_one_of=['N']), cycle())
pairs_df = compose_product(binary_combination(distance=(1,4)), cycle_df, cycle_with_N_df)
df = compose(intersection_edges(), pairs_df)
draw(graph, df)

In [ ]:
from abstractgraph.xml import register_from_module, operator_to_xml_string, operator_from_xml_string
import abstractgraph.operators as qg_ops
register_from_module(qg_ops)
xml_df = operator_to_xml_string(df, pretty=True)
print(xml_df)


In [ ]:
df_rt = operator_from_xml_string(xml_df)
draw(graph, df)

---